# TP 2 — Exercice 2 : Filtrage spatial, métrique PSNR et Devoir noté

1. Chargement de `im1` (originale), `im2` (bruit gaussien), `im3` (bruit impulsionnel).
2. Filtre moyenneur $3\times 3$ (OpenCV + fonction personnelle `filtreMoyenne`).
3. Filtre médian $3\times 3$ sur `im3` et `im2` (OpenCV + fonction personnelle `filtreMedian`).
4. Calcul et validation de la métrique **PSNR / MSE** (`cv2.PSNR` vs `calculate_psnr`).
5. Benchmark complet sur `im2` et `im3` (Moyenneur, Gaussien, Médian, Bilatéral OpenCV + `filtrebilateral` corrigé) avec PSNR et temps d'exécution.
6. **Devoir noté (a) :** Accélération d'un filtre moyenneur de grande taille (Séparabilité 1D et Image Intégrale $O(1)$/pixel).
7. **Devoir noté (b) :** Filtre moyenne adaptative (formule du cours à seuil $t$ + variante robuste aux impulsions + filtre adaptatif MMSE de Lee) et comparaison PSNR.

In [ ]:
from pathlib import Path
import math
import time
import cv2
import numpy as np
import matplotlib.pyplot as plt

def find_project_root():
    cwd = Path.cwd().resolve()
    for candidate in (cwd, cwd.parent, cwd.parent.parent):
        if (candidate / "tp1_tp2_images").is_dir():
            return candidate
    raise FileNotFoundError(
        "Impossible de trouver le dossier 'tp1_tp2_images'. "
        "Lancez Jupyter depuis la racine TAI ou depuis tp1_labs / tp2_labs."
    )

ROOT_DIR = find_project_root()
IMG_DIR = ROOT_DIR / "tp1_tp2_images"
OUT_DIR = ROOT_DIR / "tp2_labs" / "outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

def load_image(filename, flags=cv2.IMREAD_COLOR):
    path = IMG_DIR / filename
    img = cv2.imread(str(path), flags)
    if img is None:
        raise FileNotFoundError(f"Image introuvable ou illisible: {path}")
    return img

def show_images(items, cols=3, figsize_per_cell=(5, 4)):
    n = len(items)
    rows = math.ceil(n / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(cols * figsize_per_cell[0], rows * figsize_per_cell[1]))
    axes = np.atleast_1d(axes).ravel()
    for ax in axes[n:]:
        ax.axis("off")
    for ax, (title, img) in zip(axes, items):
        if img.ndim == 2:
            ax.imshow(img, cmap="gray", vmin=0, vmax=255)
        elif img.shape[2] == 3:
            ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        elif img.shape[2] == 4:
            ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGRA2RGBA))
        ax.set_title(title, fontsize=10)
        ax.axis("off")
    plt.tight_layout()
    plt.show()

print(f"ROOT_DIR : {ROOT_DIR}")
print(f"IMG_DIR  : {IMG_DIR} ({len(list(IMG_DIR.iterdir()))} images)")
print(f"OUT_DIR  : {OUT_DIR}")

In [ ]:
# 1) Chargement des images (génération déterministe si TP2_Exo1 n'a pas encore été exécuté)
im1 = load_image("tp1_tp2_cameraman_gray.bmp", cv2.IMREAD_GRAYSCALE)

gauss_path = OUT_DIR / "tp2_exo1_cameraman_gauss_var100.bmp"
sp_path = OUT_DIR / "tp2_exo1_cameraman_salt_pepper_p10.bmp"

if not gauss_path.exists():
    cv2.setRNGSeed(42)
    g_noise = np.zeros(im1.shape, dtype=np.float32)
    cv2.randn(g_noise, 0.0, 10.0)
    im2 = np.clip(im1.astype(np.float32) + g_noise, 0, 255).astype(np.uint8)
    cv2.imwrite(str(gauss_path), im2)
else:
    im2 = cv2.imread(str(gauss_path), cv2.IMREAD_GRAYSCALE)

if not sp_path.exists():
    rng = np.random.default_rng(42)
    im3 = im1.copy()
    probs = rng.random(im1.shape)
    im3[probs < 0.05] = 0
    im3[probs > 0.95] = 255
    cv2.imwrite(str(sp_path), im3)
else:
    im3 = cv2.imread(str(sp_path), cv2.IMREAD_GRAYSCALE)

print("Images chargées :", im1.shape, im2.shape, im3.shape)

In [ ]:
# Fonctions personnelles : PSNR, Moyenneur, Médian, Bilatéral (corrigé)
def calculate_psnr(original_image, degraded_image):
    orig = original_image.astype(np.float64)
    deg = degraded_image.astype(np.float64)
    mse = np.mean((orig - deg) ** 2)
    if mse == 0:
        return float("inf")
    return 10.0 * math.log10((255.0 ** 2) / mse)

def filtreMoyenne(image, taille_fenetre):
    if taille_fenetre % 2 == 0:
        raise ValueError("La taille de la fenêtre doit être impaire.")
    r = taille_fenetre // 2
    padded = np.pad(image.astype(np.float64), r, mode="reflect")
    h, w = image.shape
    acc = np.zeros((h, w), dtype=np.float64)
    for dy in range(taille_fenetre):
        for dx in range(taille_fenetre):
            acc += padded[dy:dy + h, dx:dx + w]
    return np.clip(np.round(acc / (taille_fenetre * taille_fenetre)), 0, 255).astype(np.uint8)

def filtreMedian(image, taille_fenetre):
    if taille_fenetre % 2 == 0:
        raise ValueError("La taille de la fenêtre doit être impaire.")
    r = taille_fenetre // 2
    padded = np.pad(image, r, mode="edge")
    windows = np.lib.stride_tricks.sliding_window_view(padded, (taille_fenetre, taille_fenetre))
    return np.median(windows, axis=(-2, -1)).astype(np.uint8)

def filtrebilateral(image, taille_fenetre, sigma_s, sigma_r):
    if taille_fenetre % 2 == 0:
        raise ValueError("La taille de la fenêtre doit être impaire.")
    r = taille_fenetre // 2
    h, w = image.shape
    img_f = image.astype(np.float64)
    padded = np.pad(img_f, r, mode="reflect")
    sum_weights = np.zeros((h, w), dtype=np.float64)
    sum_values = np.zeros((h, w), dtype=np.float64)
    inv_2s2 = -1.0 / (2.0 * sigma_s * sigma_s)
    inv_2r2 = -1.0 / (2.0 * sigma_r * sigma_r)
    for dy in range(-r, r + 1):
        for dx in range(-r, r + 1):
            neighbor = padded[r + dy:r + dy + h, r + dx:r + dx + w]
            spatial_w = math.exp((dy * dy + dx * dx) * inv_2s2)
            color_diff = neighbor - img_f
            color_w = np.exp((color_diff * color_diff) * inv_2r2)
            w_tot = spatial_w * color_w
            sum_weights += w_tot
            sum_values += w_tot * neighbor
    return np.clip(np.round(sum_values / np.maximum(sum_weights, 1e-12)), 0, 255).astype(np.uint8)

# 2) & 3) Application des filtres 3x3 OpenCV vs Perso
imf2 = cv2.blur(im2, (3, 3))
imfp2 = filtreMoyenne(im2, 3)

imf3 = cv2.medianBlur(im3, 3)
imfm2 = cv2.medianBlur(im2, 3)
imfmp = filtreMedian(im3, 3)

# 4) Validation PSNR (OpenCV vs calculate_psnr)
pairs = [
    ("PSNR(im1, im2 bruit gauss)", im1, im2),
    ("PSNR(im1, imf2 moyenne 3x3 sur im2)", im1, imf2),
    ("PSNR(im1, imfp2 moyenne perso sur im2)", im1, imfp2),
    ("PSNR(im1, im3 bruit S&P)", im1, im3),
    ("PSNR(im1, imf3 median 3x3 sur im3)", im1, imf3),
    ("PSNR(im1, imfmp median perso sur im3)", im1, imfmp),
    ("PSNR(im1, imfm2 median 3x3 sur im2)", im1, imfm2),
]
print(f"{'Comparaison':42s} | {'cv2.PSNR':>10s} | {'Perso':>10s}")
print("-" * 68)
for label, ref, test in pairs:
    print(f"{label:42s} | {cv2.PSNR(ref, test):8.3f} dB | {calculate_psnr(ref, test):8.3f} dB")

show_images([
    ("Originale (im1)", im1),
    ("Bruit Gaussien (im2)", im2),
    ("Moyenneur 3x3 (imf2)", imf2),
    ("Bruit Poivre & Sel (im3)", im3),
    ("Médian 3x3 sur im3 (imf3)", imf3),
    ("Médian 3x3 sur im2 (imfm2)", imfm2),
], cols=3)

In [ ]:
# 5) Benchmark complet des filtres sur im2 et im3
filters = [
    ("Moyenneur 3x3", lambda img: cv2.blur(img, (3, 3))),
    ("Moyenneur 5x5", lambda img: cv2.blur(img, (5, 5))),
    ("Moyenneur 7x7", lambda img: cv2.blur(img, (7, 7))),
    ("Gaussien 3x3, sig=2.0", lambda img: cv2.GaussianBlur(img, (3, 3), 2.0)),
    ("Gaussien 3x3, sig=1.5", lambda img: cv2.GaussianBlur(img, (3, 3), 1.5)),
    ("Gaussien 3x3, sig=1.0", lambda img: cv2.GaussianBlur(img, (3, 3), 1.0)),
    ("Gaussien 3x3, sig=0.5", lambda img: cv2.GaussianBlur(img, (3, 3), 0.5)),
    ("Gaussien 5x5, sig=2.0", lambda img: cv2.GaussianBlur(img, (5, 5), 2.0)),
    ("Gaussien 5x5, sig=1.5", lambda img: cv2.GaussianBlur(img, (5, 5), 1.5)),
    ("Gaussien 5x5, sig=1.0", lambda img: cv2.GaussianBlur(img, (5, 5), 1.0)),
    ("Gaussien 5x5, sig=0.5", lambda img: cv2.GaussianBlur(img, (5, 5), 0.5)),
    ("Médian 3x3", lambda img: cv2.medianBlur(img, 3)),
    ("Médian 5x5", lambda img: cv2.medianBlur(img, 5)),
    ("Médian 7x7", lambda img: cv2.medianBlur(img, 7)),
    ("Bilatéral OpenCV (d=7,40,40)", lambda img: cv2.bilateralFilter(img, 7, 40, 40)),
    ("Bilatéral Perso (7, 2.0, 40)", lambda img: filtrebilateral(img, 7, 2.0, 40.0)),
]

print(f"Baseline -> im2 (Gauss): {calculate_psnr(im1, im2):.2f} dB | im3 (S&P): {calculate_psnr(im1, im3):.2f} dB\n")
print(f"{'Filtre':30s} | {'PSNR im2 (dB)':>13s} | {'Temps im2 (ms)':>14s} | {'PSNR im3 (dB)':>13s} | {'Temps im3 (ms)':>14s}")
print("-" * 96)

best_im2 = (-1.0, "")
best_im3 = (-1.0, "")
for name, fn in filters:
    t0 = time.perf_counter()
    out2 = fn(im2)
    dt2 = (time.perf_counter() - t0) * 1000.0
    p2 = calculate_psnr(im1, out2)

    t0 = time.perf_counter()
    out3 = fn(im3)
    dt3 = (time.perf_counter() - t0) * 1000.0
    p3 = calculate_psnr(im1, out3)

    if p2 > best_im2[0]:
        best_im2 = (p2, name)
    if p3 > best_im3[0]:
        best_im3 = (p3, name)
    print(f"{name:30s} | {p2:13.3f} | {dt2:14.3f} | {p3:13.3f} | {dt3:14.3f}")

print(f"\nMeilleur sur im2 (Gaussien)    : {best_im2[1]} ({best_im2[0]:.3f} dB)")
print(f"Meilleur sur im3 (Poivre & Sel): {best_im3[1]} ({best_im3[0]:.3f} dB)")

### Devoir noté (a) : Accélération d'un filtre moyenneur de grande taille

Pour un masque de taille $k \times k$ sur une image $H \times W$ :
1. **Approche naïve :** $O(H \cdot W \cdot k^2)$ opérations.
2. **Approche séparable (1D horizontal puis 1D vertical) :** $O(H \cdot W \cdot 2k)$ opérations (propriété de séparabilité du cours).
3. **Approche par Image Intégrale (Summed-Area Table) :** $O(H \cdot W)$ indépendant de la taille $k$ (seulement 4 lectures par pixel quelle que soit la taille du filtre !).

In [ ]:
def filtreMoyenne_naif_boucles(image, k):
    r = k // 2
    h, w = image.shape
    padded = np.pad(image.astype(np.float64), r, mode="reflect")
    out = np.zeros((h, w), dtype=np.uint8)
    area = float(k * k)
    for i in range(h):
        for j in range(w):
            out[i, j] = int(round(np.sum(padded[i:i + k, j:j + k]) / area))
    return out

def filtreMoyenne_separable(image, k):
    r = k // 2
    h, w = image.shape
    padded = np.pad(image.astype(np.float64), r, mode="reflect")
    # Passe horizontale 1D
    horiz = np.zeros((h + 2 * r, w), dtype=np.float64)
    for dx in range(k):
        horiz += padded[:, dx:dx + w]
    horiz /= k
    # Passe verticale 1D
    vert = np.zeros((h, w), dtype=np.float64)
    for dy in range(k):
        vert += horiz[dy:dy + h, :]
    vert /= k
    return np.clip(np.round(vert), 0, 255).astype(np.uint8)

def filtreMoyenne_integrale(image, k):
    r = k // 2
    h, w = image.shape
    padded = np.pad(image.astype(np.float64), r, mode="reflect")
    sat = np.zeros((padded.shape[0] + 1, padded.shape[1] + 1), dtype=np.float64)
    sat[1:, 1:] = np.cumsum(np.cumsum(padded, axis=0), axis=1)
    sums = sat[k:, k:] - sat[:-k, k:] - sat[k:, :-k] + sat[:-k, :-k]
    return np.clip(np.round(sums / (k * k)), 0, 255).astype(np.uint8)

print(f"{'Taille':>8s} | {'Naïf (ms)':>11s} | {'Séparable (ms)':>15s} | {'Intégrale O(1) (ms)':>19s} | {'Accélération':>13s} | {'Max diff':>8s}")
print("-" * 88)
for k in (7, 15, 25, 35):
    # Naïf mesuré sur 7 et 15 (extrapolé ou mesuré directement car 256x256 reste rapide)
    t0 = time.perf_counter()
    ref = filtreMoyenne_naif_boucles(im2, k)
    dt_naif = (time.perf_counter() - t0) * 1000.0

    t0 = time.perf_counter()
    res_sep = filtreMoyenne_separable(im2, k)
    dt_sep = (time.perf_counter() - t0) * 1000.0

    t0 = time.perf_counter()
    res_int = filtreMoyenne_integrale(im2, k)
    dt_int = (time.perf_counter() - t0) * 1000.0

    max_diff = int(np.max(np.abs(ref.astype(np.int16) - res_int.astype(np.int16))))
    speedup = dt_naif / max(dt_int, 1e-6)
    print(f"{f'{k}x{k}':>8s} | {dt_naif:11.2f} | {dt_sep:15.3f} | {dt_int:19.3f} | {speedup:11.1f}x | {max_diff:8d}")

### Devoir noté (b) : Filtre moyenne adaptative et comparaison PSNR

1. **Formule du cours (Slide 42) :**
   $$C[i, j] = \frac{\sum_{k=1}^{L} w(|a_k - A[i,j]|)\, a_k}{\sum_{k=1}^{L} w(|a_k - A[i,j]|)}, \qquad w(x) = \begin{cases} 1 & \text{si } |x| \le t \\ 0 & \text{sinon} \end{cases}$$
2. **Variante robuste aux impulsions :** si seul le pixel central vérifie $|a_k - A[i,j]| \le t$ (cas typique d'un pixel poivre/sel isolé), on remplace par la médiane locale du voisinage.
3. **Filtre adaptatif MMSE local (Lee) pour bruit gaussien :** adapte le poids du lissage au rapport entre la variance du bruit $\sigma_n^2$ et la variance locale $\sigma_L^2$.

In [ ]:
def filtreMoyenneAdaptative_cours(image, k=5, seuil=30.0, robust_impulsion=False):
    r = k // 2
    h, w = image.shape
    center = image.astype(np.float64)
    padded = np.pad(center, r, mode="reflect")
    windows = np.lib.stride_tricks.sliding_window_view(padded, (k, k))
    diffs = np.abs(windows - center[:, :, None, None])
    weights = (diffs <= seuil).astype(np.float64)
    sum_w = np.sum(weights, axis=(-2, -1))
    sum_val = np.sum(weights * windows, axis=(-2, -1))
    out = sum_val / np.maximum(sum_w, 1.0)
    if robust_impulsion:
        # Si le pixel central est isolé (<= 2 voisins similaires dans la fenêtre), on prend la médiane locale
        isolated = sum_w <= 2.0
        if np.any(isolated):
            med = np.median(windows, axis=(-2, -1))
            out = np.where(isolated, med, out)
    return np.clip(np.round(out), 0, 255).astype(np.uint8)

def filtreMoyenneAdaptative_MMSE(image, k=5, noise_var=100.0):
    r = k // 2
    img_f = image.astype(np.float64)
    padded = np.pad(img_f, r, mode="reflect")
    windows = np.lib.stride_tricks.sliding_window_view(padded, (k, k))
    local_mean = np.mean(windows, axis=(-2, -1))
    local_var = np.var(windows, axis=(-2, -1))
    gain = np.clip(noise_var / np.maximum(local_var, noise_var), 0.0, 1.0)
    out = img_f - gain * (img_f - local_mean)
    return np.clip(np.round(out), 0, 255).astype(np.uint8)

comp_rows = [
    ("Bruité (référence)", im2, im3),
    ("Moyenneur 3x3", cv2.blur(im2, (3, 3)), cv2.blur(im3, (3, 3))),
    ("Gaussien 3x3 sig=1.0", cv2.GaussianBlur(im2, (3, 3), 1.0), cv2.GaussianBlur(im3, (3, 3), 1.0)),
    ("Médian 3x3", cv2.medianBlur(im2, 3), cv2.medianBlur(im3, 3)),
    ("Bilatéral (d=7,40,40)", cv2.bilateralFilter(im2, 7, 40, 40), cv2.bilateralFilter(im3, 7, 40, 40)),
    ("Moyenne Adaptative Cours (5x5, t=30)", filtreMoyenneAdaptative_cours(im2, 5, 30, False),
     filtreMoyenneAdaptative_cours(im3, 5, 30, False)),
    ("Moyenne Adaptative Robuste (5x5, t=30)", filtreMoyenneAdaptative_cours(im2, 5, 30, True),
     filtreMoyenneAdaptative_cours(im3, 5, 30, True)),
    ("Moyenne Adaptative MMSE Lee (5x5)", filtreMoyenneAdaptative_MMSE(im2, 5, 100.0),
     filtreMoyenneAdaptative_MMSE(im3, 5, 100.0)),
]

print(f"{'Méthode':40s} | {'PSNR im2 (Gauss)':>16s} | {'PSNR im3 (S&P)':>14s}")
print("-" * 76)
for label, r2, r3 in comp_rows:
    print(f"{label:40s} | {calculate_psnr(im1, r2):14.3f} dB | {calculate_psnr(im1, r3):12.3f} dB")

show_images([
    ("Adaptative Cours im2", comp_rows[5][1]),
    ("Adaptative MMSE im2", comp_rows[7][1]),
    ("Adaptative Cours im3", comp_rows[5][2]),
    ("Adaptative Robuste im3", comp_rows[6][2]),
], cols=4)